In [4]:
# Load packages
suppressPackageStartupMessages({
    library(ggplot2)
    library(biomaRt)
    library(BiocParallel)
    library(Matrix)
    library(matrixStats)
    library(scater)
    library(reshape2)
    library(knitr)
    library(scran)
    library(batchelor)
    library(reticulate)
    library(SingleCellExperiment)
    library(data.table)
    library(dplyr)
    library(gridExtra)
})
    ncores = 4
    mcparam = MulticoreParam(workers = ncores)
    register(mcparam)
    BPPARAM = SerialParam()

options(repr.plot.width=15, repr.plot.height=8)

Warning message:
“package ‘knitr’ was built under R version 4.1.2”


In [5]:
main = "/rds/project/rds-SDzz0CATGms/users/bt392/03_Stat3_RNA/"
out_dir = paste0(main, '07_atlas_mapping/')
plot_dir = paste0(main, '07_atlas_mapping/plots/')
dir.create(out_dir, showWarnings = FALSE)
dir.create(plot_dir, showWarnings = FALSE)

atlas_in = '/rds/project/rds-SDzz0CATGms/users/bt392/mouse/Mixl1_KO/atlas/atlas/'
source(paste0(main, "core_functions.R"))

In [74]:
source(paste0(main, "mapping_functions_extended.R"))

In [55]:
load_data(normalise=TRUE)
sce_query = sce[Matrix::rowSums(counts(sce)) > 0,]
meta_query = meta

In [123]:
sce_atlas = readRDS('/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation/pijuansala2019_gastrulation10x/processed/SingleCellExperiment.rds')

In [124]:
atlas_meta = fread('/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation/pijuansala2019_gastrulation10x/sample_metadata.txt.gz')

In [125]:
atlas_meta = atlas_meta[sample(nrow(atlas_meta), 2000),]
sce_atlas = sce_atlas[,atlas_meta$cell]

In [63]:
meta_query = meta_query[meta_query$cell %in% colnames(sce_query),]
meta_query = meta_query[sample(nrow(meta_query), 2000),]
sce_query = sce_query[,meta_query$cell]

In [126]:
clusters <- scran::quickCluster(sce_atlas)
sce_atlas <- scran::computeSumFactors(sce_atlas, cluster = clusters)
sce_atlas <- scuttle::logNormCounts(sce_atlas)

In [127]:
# genes shared across datasets
shared_genes = merge(data.frame('V1'=names(sce_atlas)), data.frame('V1'=names(sce_query)), by='V1')

# filter cells & genes chimera dataset
sce_query = sce_query[shared_genes$V1,!c(meta_query$doublet | meta_query$stripped)]
meta_query = meta_query[!c(meta_query$doublet | meta_query$stripped),]

# filter cells & genes atlas dataset
sce_atlas = sce_atlas[shared_genes$V1,]

In [113]:
mapping = mapWrap(
        atlas_sce = sce_atlas, 
        atlas_meta = atlas_meta,
        map_sce = sce_query, 
        map_meta = meta_query, 
        # genes = marker_genes, 
        npcs = 10, 
        k = 15)

Normalizing joint dataset...

Done


Genes not provided. Computing highly variable genes...

Warning message in regularize.values(x, y, ties, missing(ties), na.rm = na.rm):
“collapsing to unique 'x' values”
Done


Performing PCA...

Done


Batch effect correction for the atlas...

Done


MNN mapping...

Warning message in .refine_k(k, precomputed, query = TRUE):
“'k' capped at the number of observations”


ERROR: Error in t.default(apply(k.mapped, 1, function(x) atlas_meta$celltype[match(x, : argument is not a matrix


In [118]:
atlas_meta = atlas_meta[order(match(atlas_meta$cell, rownames(sce_atlas)))]

In [128]:
atlas_sce = sce_atlas
atlas_meta = atlas_meta
map_sce = sce_query
map_meta = meta_query
# genes = marker_genes, 
npcs = 10
k = 15

In [129]:
genes = NULL

In [130]:
   
  message("Normalizing joint dataset...")
  
  #easier to avoid directly binding sce objects as it is a lot more likely to have issues
  sce_all <- SingleCellExperiment::SingleCellExperiment(
    list(counts=Matrix::Matrix(cbind(counts(atlas_sce),counts(map_sce)),sparse=TRUE)))
  
  #big_sce <- scater::normalize(sce_all)
  #big_sce <- scater::logNormCounts(sce_all) # edited 09.02. because normalize deprecated in favour of logNormCounts
  big_sce <- multiBatchNorm(sce_all, batch=c(atlas_meta$sample, map_meta$sample)) # edited 17.02. because now multibatchnorm exists
  message("Done\n")
  
  if (is.null(genes)) {
    message("Genes not provided. Computing highly variable genes...")
    hvgs <- getHVGs(big_sce, block=c(atlas_meta$sample, map_meta$sample))
    message("Done\n")
  } else {
    hvgs <- genes
    message(sprintf("%d Genes provided...",length(genes)))
  }
head(atlas_meta)  
  message("Performing PCA...")
  big_pca <- multiBatchPCA(big_sce,
                           batch=c(atlas_meta$sample, map_meta$sample),
                           subset.row = hvgs,
                           d = npcs,
                           preserve.single = TRUE,
                           assay.type = "logcounts")[[1]]
  rownames(big_pca) <- colnames(big_sce) 
  atlas_pca <- big_pca[1:ncol(atlas_sce),]
  map_pca   <- big_pca[-(1:ncol(atlas_sce)),]
  message("Done\n")
head(atlas_meta)  
  message("Batch effect correction for the atlas...")  
  order_df        <- atlas_meta[!duplicated(atlas_meta$sample), c("stage", "sample")]
  order_df$ncells <- sapply(order_df$sample, function(x) sum(atlas_meta$sample == x))
                                         
  order_df$stage  <- factor(order_df$stage, 
                        levels = rev(c("E9.5",
                                       "E9.25",
                                       "E9.0",
                                       "E8.75",
                                       "E8.5",
                                       "E8.25",
                                       "E8.0",
                                       "E7.75",
                                       "E7.5",
                                       "E7.25",
                                       "mixed_gastrulation",
                                       "E7.0",
                                       "E6.75",
                                       "E6.5")))
                            
  order_df       <- order_df[order(order_df$stage, order_df$ncells, decreasing = TRUE),]
  order_df$stage <- as.character(order_df$stage)
head(atlas_meta)
  message('for real')
  set.seed(42)
  atlas_corrected <- doBatchCorrect(counts         = logcounts(atlas_sce[hvgs,]), 
                                    timepoints      = atlas_meta$stage, 
                                    samples         = atlas_meta$sample, 
                                    timepoint_order = order_df$stage, 
                                    sample_order    = order_df$sample, 
                                    pc_override     = atlas_pca,
                                    npc             = npcs)
  message("Done\n")
 
  

Normalizing joint dataset...

Done


Genes not provided. Computing highly variable genes...

Warning message in regularize.values(x, y, ties, missing(ties), na.rm = na.rm):
“collapsing to unique 'x' values”
Warning message in regularize.values(x, y, ties, missing(ties), na.rm = na.rm):
“collapsing to unique 'x' values”
Done




cell,barcode,sample,stage,sequencing.batch,doublet,stripped,celltype,umapX,umapY,nFeature_RNA,nCount_RNA
<chr>,<chr>,<int>,<chr>,<int>,<lgl>,<lgl>,<chr>,<dbl>,<dbl>,<int>,<int>
cell_33938,CTTGAACTCGTAGT,16,E8.0,2,FALSE,FALSE,ExE_ectoderm,9.33207489,-5.485074,3842,16755
cell_115888,ATCACTACCCCTAC,33,E8.0,3,FALSE,FALSE,Visceral_endoderm,-4.52211920,-12.544966,3053,11139
cell_98574,CCAGACCTTTTGGG,29,E8.5,3,FALSE,FALSE,Paraxial_mesoderm,3.90397731,6.554718,3179,13407
cell_23146,ATCATGCTGGTGTT,13,E7.75,2,FALSE,FALSE,Intermediate_mesoderm,0.01567323,3.516167,3655,16722
cell_854,CACACCTGCTTACT,3,E7.5,1,FALSE,FALSE,Rostral_neurectoderm,-5.95494283,-2.728000,2254,6155
cell_5581,AATTCCTGCGAGAG,7,E6.75,1,FALSE,FALSE,Mixed_mesoderm,-2.99806803,7.876060,2335,7848


Performing PCA...

Done




cell,barcode,sample,stage,sequencing.batch,doublet,stripped,celltype,umapX,umapY,nFeature_RNA,nCount_RNA
<chr>,<chr>,<int>,<chr>,<int>,<lgl>,<lgl>,<chr>,<dbl>,<dbl>,<int>,<int>
cell_33938,CTTGAACTCGTAGT,16,E8.0,2,FALSE,FALSE,ExE_ectoderm,9.33207489,-5.485074,3842,16755
cell_115888,ATCACTACCCCTAC,33,E8.0,3,FALSE,FALSE,Visceral_endoderm,-4.52211920,-12.544966,3053,11139
cell_98574,CCAGACCTTTTGGG,29,E8.5,3,FALSE,FALSE,Paraxial_mesoderm,3.90397731,6.554718,3179,13407
cell_23146,ATCATGCTGGTGTT,13,E7.75,2,FALSE,FALSE,Intermediate_mesoderm,0.01567323,3.516167,3655,16722
cell_854,CACACCTGCTTACT,3,E7.5,1,FALSE,FALSE,Rostral_neurectoderm,-5.95494283,-2.728000,2254,6155
cell_5581,AATTCCTGCGAGAG,7,E6.75,1,FALSE,FALSE,Mixed_mesoderm,-2.99806803,7.876060,2335,7848


Batch effect correction for the atlas...



cell,barcode,sample,stage,sequencing.batch,doublet,stripped,celltype,umapX,umapY,nFeature_RNA,nCount_RNA
<chr>,<chr>,<int>,<chr>,<int>,<lgl>,<lgl>,<chr>,<dbl>,<dbl>,<int>,<int>
cell_33938,CTTGAACTCGTAGT,16,E8.0,2,FALSE,FALSE,ExE_ectoderm,9.33207489,-5.485074,3842,16755
cell_115888,ATCACTACCCCTAC,33,E8.0,3,FALSE,FALSE,Visceral_endoderm,-4.52211920,-12.544966,3053,11139
cell_98574,CCAGACCTTTTGGG,29,E8.5,3,FALSE,FALSE,Paraxial_mesoderm,3.90397731,6.554718,3179,13407
cell_23146,ATCATGCTGGTGTT,13,E7.75,2,FALSE,FALSE,Intermediate_mesoderm,0.01567323,3.516167,3655,16722
cell_854,CACACCTGCTTACT,3,E7.5,1,FALSE,FALSE,Rostral_neurectoderm,-5.95494283,-2.728000,2254,6155
cell_5581,AATTCCTGCGAGAG,7,E6.75,1,FALSE,FALSE,Mixed_mesoderm,-2.99806803,7.876060,2335,7848


for real

Warning message in .refine_k(k, precomputed, query = TRUE):
“'k' capped at the number of observations”
Warning message in .refine_k(k, precomputed, query = TRUE):
“'k' capped at the number of observations”
Warning message in .refine_k(k, precomputed, query = TRUE):
“'k' capped at the number of observations”
Warning message in .refine_k(k, precomputed, query = TRUE):
“'k' capped at the number of observations”
Warning message in .refine_k(k, precomputed, query = TRUE):
“'k' capped at the number of observations”
Done




In [131]:
head(atlas_meta)

cell,barcode,sample,stage,sequencing.batch,doublet,stripped,celltype,umapX,umapY,nFeature_RNA,nCount_RNA
<chr>,<chr>,<int>,<chr>,<int>,<lgl>,<lgl>,<chr>,<dbl>,<dbl>,<int>,<int>
cell_33938,CTTGAACTCGTAGT,16,E8.0,2,FALSE,FALSE,ExE_ectoderm,9.33207489,-5.485074,3842,16755
cell_115888,ATCACTACCCCTAC,33,E8.0,3,FALSE,FALSE,Visceral_endoderm,-4.52211920,-12.544966,3053,11139
cell_98574,CCAGACCTTTTGGG,29,E8.5,3,FALSE,FALSE,Paraxial_mesoderm,3.90397731,6.554718,3179,13407
cell_23146,ATCATGCTGGTGTT,13,E7.75,2,FALSE,FALSE,Intermediate_mesoderm,0.01567323,3.516167,3655,16722
cell_854,CACACCTGCTTACT,3,E7.5,1,FALSE,FALSE,Rostral_neurectoderm,-5.95494283,-2.728000,2254,6155
cell_5581,AATTCCTGCGAGAG,7,E6.75,1,FALSE,FALSE,Mixed_mesoderm,-2.99806803,7.876060,2335,7848


In [ ]:
order=NULL

In [132]:
 
  message("MNN mapping...")                        
  correct <- reducedMNN(rbind(atlas_corrected, map_pca),
                      batch=c(rep("ATLAS", dim(atlas_meta)[1]), map_meta$sample))$corrected

MNN mapping...

Warning message in .refine_k(k, precomputed, query = TRUE):
“'k' capped at the number of observations”


In [134]:

  atlas   <- 1:nrow(atlas_pca)
  correct_atlas <- correct[atlas,]
  correct_map   <- correct[-atlas,]


In [135]:
head(atlas_meta)

cell,barcode,sample,stage,sequencing.batch,doublet,stripped,celltype,umapX,umapY,nFeature_RNA,nCount_RNA
<chr>,<chr>,<int>,<chr>,<int>,<lgl>,<lgl>,<chr>,<dbl>,<dbl>,<int>,<int>
cell_33938,CTTGAACTCGTAGT,16,E8.0,2,FALSE,FALSE,ExE_ectoderm,9.33207489,-5.485074,3842,16755
cell_115888,ATCACTACCCCTAC,33,E8.0,3,FALSE,FALSE,Visceral_endoderm,-4.52211920,-12.544966,3053,11139
cell_98574,CCAGACCTTTTGGG,29,E8.5,3,FALSE,FALSE,Paraxial_mesoderm,3.90397731,6.554718,3179,13407
cell_23146,ATCATGCTGGTGTT,13,E7.75,2,FALSE,FALSE,Intermediate_mesoderm,0.01567323,3.516167,3655,16722
cell_854,CACACCTGCTTACT,3,E7.5,1,FALSE,FALSE,Rostral_neurectoderm,-5.95494283,-2.728000,2254,6155
cell_5581,AATTCCTGCGAGAG,7,E6.75,1,FALSE,FALSE,Mixed_mesoderm,-2.99806803,7.876060,2335,7848


In [147]:
get_meta <- function(correct_atlas, atlas_meta, correct_map, map_meta, k_map = 10){
  knns <- BiocNeighbors::queryKNN(correct_atlas, correct_map, k = k_map, get.index = TRUE,
    get.distance = FALSE)
  #get closest k matching cells
  k.mapped  <- t(apply(knns$index, 1, function(x) atlas_meta$cell[x]))
  celltypes <- t(apply(k.mapped, 1, function(x) atlas_meta$celltype[match(x, atlas_meta$cell)]))
                       
#  celltype_originals <- t(apply(k.mapped, 1, function(x) atlas_meta$celltype_original[match(x, atlas_meta$cell)]))
                       
  stages    <- t(apply(k.mapped, 1, function(x) atlas_meta$stage[match(x, atlas_meta$cell)]))
  celltype.mapped <- apply(celltypes, 1, function(x) getmode(x, 1:length(x)))
                             
#  celltype_original.mapped <- apply(celltype_originals, 1, function(x) getmode(x, 1:length(x)))
                           
  stage.mapped    <- apply(stages, 1, function(x) getmode(x, 1:length(x)))
  out <- lapply(1:length(celltype.mapped), function(x){
    list(cells.mapped     = k.mapped[x,],
         celltype.mapped  = celltype.mapped[x],
         
#         celltype_original.mapped  = celltype_original.mapped[x],

         stage.mapped     = stage.mapped[x],
         celltypes.mapped = celltypes[x,],
         
 #        celltype_originals.mapped = celltype_originals[x,],

         stages.mapped    = stages[x,])
  })
  names(out) <- map_meta$cell
  return(out)  
}

In [148]:

  mapping <- get_meta(correct_atlas = correct_atlas,
                      atlas_meta = atlas_meta,
                      correct_map = correct_map,
                      map_meta = map_meta,
                      k_map = k)
  message("Done\n")


Done




In [153]:
getMappingScore <- function(mapping){
    out <- list()
    celltypes_accrossK <- matrix(unlist(mapping$celltypes.mapped),
                                 nrow=length(mapping$celltypes.mapped[[1]]),
                                 ncol=length(mapping$celltypes.mapped))
    
#    celltype_originals_accrossK <- matrix(unlist(mapping$celltype_originals.mapped), # ADDED
#                                  nrow=length(mapping$celltype_originals.mapped[[1]]),
#                                  ncol=length(mapping$celltype_originals.mapped))
    
    cellstages_accrossK <- matrix(unlist(mapping$cellstages.mapped),
                                  nrow=length(mapping$cellstages.mapped[[1]]),
                                  ncol=length(mapping$cellstages.mapped))
    out$celltype.score <- NULL
    for (i in 1:nrow(celltypes_accrossK)){
        p <- max(table(celltypes_accrossK[i,]))
        index <- which(table(celltypes_accrossK[i,]) == p)
        p <- p/length(mapping$celltypes.mapped)
        out$celltype.score <- c(out$celltype.score,p)
    }
    
    # out$celltype_original.score <- NULL # ADDED
    # for (i in 1:nrow(celltype_originals_accrossK)){
    #     p <- max(table(celltype_originals_accrossK[i,]))
    #     index <- which(table(celltype_originals_accrossK[i,]) == p)
    #     p <- p/length(mapping$celltype_originals.mapped)
    #     out$celltype_original.score <- c(out$celltype_original.score,p)
    # }
    
    out$cellstage.score <- NULL
    for (i in 1:nrow(cellstages_accrossK)){
        p <- max(table(cellstages_accrossK[i,]))
        index <- which(table(cellstages_accrossK[i,]) == p)
        p <- p/length(mapping$cellstages.mapped)
        out$cellstage.score <- c(out$cellstage.score,p)
    }
    return(out)  
}

In [154]:

  message("Computing mapping scores...") 
  out <- list()
  for (i in seq(from = 1, to = k)) {
    out$closest.cells[[i]]     <- sapply(mapping, function(x) x$cells.mapped[i])
    out$celltypes.mapped[[i]]  <- sapply(mapping, function(x) x$celltypes.mapped[i])
                                         
#    out$celltype_originals.mapped[[i]]  <- sapply(mapping, function(x) x$celltype_originals.mapped[i])

    out$cellstages.mapped[[i]] <- sapply(mapping, function(x) x$stages.mapped[i])
  }  
  multinomial.prob <- getMappingScore(out)
  message("Done\n")
  
  message("Writing output...") 
  out$correct_atlas <- correct_atlas
  out$correct_map <- correct_map
  ct <- sapply(mapping, function(x) x$celltype.mapped); is.na(ct) <- lengths(ct) == 0
               
 # cto <- sapply(mapping, function(x) x$celltype_original.mapped); is.na(cto) <- lengths(cto) == 0

  st <- sapply(mapping, function(x) x$stage.mapped); is.na(st) <- lengths(st) == 0
  cm <- sapply(mapping, function(x) x$cells.mapped[1]); is.na(cm) <- lengths(cm) == 0
  out$mapping <- data.frame(
      cell            = names(mapping), 
      celltype.mapped = unlist(ct),
      
#      celltype_original.mapped = unlist(cto),

      stage.mapped    = unlist(st),
      closest.cell    = unlist(cm))
  
  out$mapping <- cbind(out$mapping,multinomial.prob)
  out$pca <- big_pca
  message("Done\n")

Computing mapping scores...

Done


Writing output...

Done




In [155]:
str(out)

List of 7
 $ closest.cells    :List of 15
  ..$ : Named chr [1:1688] "cell_24866" "cell_49103" "cell_100300" "cell_90009" ...
  .. ..- attr(*, "names")= chr [1:1688] "cell_26953" "cell_20312" "cell_23648" "cell_5763" ...
  ..$ : Named chr [1:1688] "cell_12041" "cell_114696" "cell_114774" "cell_5669" ...
  .. ..- attr(*, "names")= chr [1:1688] "cell_26953" "cell_20312" "cell_23648" "cell_5763" ...
  ..$ : Named chr [1:1688] "cell_20001" "cell_49899" "cell_101556" "cell_84501" ...
  .. ..- attr(*, "names")= chr [1:1688] "cell_26953" "cell_20312" "cell_23648" "cell_5763" ...
  ..$ : Named chr [1:1688] "cell_24871" "cell_12648" "cell_125775" "cell_84366" ...
  .. ..- attr(*, "names")= chr [1:1688] "cell_26953" "cell_20312" "cell_23648" "cell_5763" ...
  ..$ : Named chr [1:1688] "cell_36801" "cell_125775" "cell_73588" "cell_133652" ...
  .. ..- attr(*, "names")= chr [1:1688] "cell_26953" "cell_20312" "cell_23648" "cell_5763" ...
  ..$ : Named chr [1:1688] "cell_17549" "cell_7392" "cell_1024

In [158]:
head(out$correct_atlas    )
head(out$correct_map    )

cell_33938,-11.35652,-9.4809431,-8.082906,3.8486635,6.1031954,-6.5336603,0.2287074,-3.0647775,0.08069403,2.323265
cell_115888,-12.55167,-2.9727075,-2.439228,-0.3205593,-4.6941958,-1.1549225,-3.4349995,-2.9297206,7.35286023,2.780560
cell_98574,-14.18758,0.2439005,-3.551460,0.1821573,-4.1381036,-2.9155875,1.6031302,-1.6115020,-1.84495788,2.608319
cell_23146,-15.33996,-0.5240280,-3.290284,-1.6356515,-3.4036231,-4.1134289,5.1031340,0.3109273,1.97318466,-2.313008
cell_854,-13.17119,-0.7277751,-3.716714,1.5846387,-4.4879436,-2.2095373,-1.2177608,-2.4951882,0.96232125,1.928364
cell_5581,-15.46749,-3.9495475,-3.403595,-4.6579828,-0.7915832,-0.1178495,4.0801243,0.5909452,0.76963901,-2.123129


cell_26953,-15.96810,-7.4139034,-0.04641255,-4.3472737,3.718321,4.5852344,1.318450,-3.794229,1.0955446,0.007016576
cell_20312,-14.90310,-1.7200140,-3.22147819,1.8789930,-1.262978,-1.6879663,3.127827,-2.313609,-1.7801145,-2.635982706
cell_23648,-15.78524,-3.7321572,-1.83988181,-1.4173556,-0.246332,1.1412749,3.594182,-1.865100,-0.3780978,-3.809046562
cell_5763,-14.43971,-0.9392468,-2.53824339,3.4481681,-2.032241,-2.5536499,1.781757,-3.736667,-1.3712315,-0.527781257
cell_24329,-14.88995,-5.3477074,-2.02851046,-5.8072000,-1.197853,4.4194381,-1.403876,-1.104025,3.0594223,-0.841498415
cell_44528,-15.95089,-1.9948859,-0.70635458,0.2290526,-1.213279,0.8279702,3.029169,-3.062697,1.2338394,-3.924070888


In [159]:
nrow(out$pca)

[1] 3688

In [ ]:
head(mapping$pca)